In [1]:
import scanpy as sc
import scvi
import numpy as np
import sys
sys.path.append('../')

from scripts.subset_hvg import subset_to_hvg

adata = sc.read_h5ad(
    "../../data/obesity_challenge_2.h5ad"
)

print(adata)

adata_subset, hvg_genes,sig_genes = subset_to_hvg(
    adata,
    hvg_path="../../data/preprocessed/HVG/hvg10000_genes.txt",
    include_signature_genes=True
)

print(adata_subset.var_names)

/opt/anaconda3/envs/X-Cells/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AnnData object with n_obs × n_vars = 90815 × 36601
    obs: 'nCount_RNA', 'nFeature_RNA', 'nCount_guide', 'nFeature_guide', 'percent.mt', 'SampleID', 'Day', 'num_features', 'feature_call', 'num_umis', 'gene', 'adipo', 'pre_adipo', 'other', 'lipo'
    uns: 'log1p'
    layers: 'counts'
HVG requested: 10000
Signature genes requested: 820
Signature genes used: 481
Total genes used: 10481
Index(['MIR1302-2HG', 'AL627309.1', 'AL669831.2', 'AL645608.4', 'SAMD11',
       'PLEKHN1', 'HES4', 'AL645608.1', 'RNF223', 'SDF4',
       ...
       'LAMB3', 'LRG1', 'MB', 'NAPEPLD', 'NUDT7', 'PEX11A', 'RREB1', 'SLC2A4',
       'SOX13', 'VSTM2A'],
      dtype='object', name='gene', length=10481)


In [2]:
import sys
sys.path.append('../')
from scripts.pairing import assign_state_label
adata_subset.obs["cell_state"] = assign_state_label(
    adata_subset.obs
)
print(adata_subset.obs["cell_state"].value_counts())

The history saving thread hit an unexpected error (OperationalError('unable to open database file')).History will not be written to the database.
cell_state
other         43319
pre_adipo     26681
adipo         16057
lipo_adipo     4758
Name: count, dtype: int64


# Scvi

In [ ]:
import scvi

# setup anndata
scvi.model.SCVI.setup_anndata(
    adata_subset,
    layer="counts"
)

# 先訓練 unsupervised VAE
vae = scvi.model.SCVI(
    adata_subset,
    n_latent=50
)

vae.train(
    max_epochs=200,
    accelerator="mps",
    devices=1,
    batch_size=512,
    early_stopping=True
)

# 再轉成 semi-supervised model
scanvi = scvi.model.SCANVI.from_scvi_model(
    vae,
    labels_key="cell_state",
    unlabeled_category="unknown"   # 必須是字串
)

scanvi.train(
    max_epochs=200,
    accelerator="gpu",
    devices=1,
    batch_size=512,
    early_stopping=True
)

# 取得 latent embedding
adata_subset.obsm["X_scanvi"] = scanvi.get_latent_representation(adata_subset)

print(adata_subset.obsm["X_scanvi"].shape)

/opt/anaconda3/envs/X-Cells/lib/python3.11/site-packages/scvi/train/_trainrunner.py:98: UserWarning: `accelerator` has been set to `mps`. Please note that not all PyTorch/Jax operations are supported with this backend. as a result, some models might be slower and less accurate than usual. Please verify your analysis!Refer to https://github.com/pytorch/pytorch/issues/77764 for more details.
  accelerator, lightning_devices, device = parse_device_args(
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/opt/anaconda3/envs/X-Cells/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/opt/anaconda3/envs

Epoch 200/200: 100%|██████████| 200/200 [11:12<00:00,  3.27s/it, v_num=1, train_loss=3.79e+3]

`Trainer.fit` stopped: `max_epochs=200` reached.


Epoch 200/200: 100%|██████████| 200/200 [11:12<00:00,  3.36s/it, v_num=1, train_loss=3.79e+3]


ValueError: Categorical categories cannot be null

In [15]:
save_dir = "model/scanvi_model"

scanvi.save(
    save_dir,
    overwrite=True
)

print("model saved to:", save_dir)

model saved to: model/scanvi_model
